# V38｜防泄漏验证 + AE 恢复目标消融（A/B/C）

**目的**：保持 V33 的双视图共享 AE(512→128)、SelfExpr、OwnOnly、GAT-Residual 和 GeneRead 与 V32 邻居筛选，只测试 AE 的恢复目标。

- **A**：旧版全矩阵 count MSE + CL（包含未知零作为重构目标）。
- **B**：仅在人工遮盖的已观测非零位置计算 MSE + CL。
- **C**：B + 少量未遮盖已观测非零位置重构 MSE + CL（仍不监督未知零）。

**严格协议**：在 AE / SelfExpr / OwnOnly / GAT 之前，从 dropout40 的已观测非零中抽出固定验证位置，**只对遮盖后的训练矩阵**拟合整个三阶段流水线；A/B/C 共用同一个验证掩码与同一份筛选机制。只根据独立人工遮盖验证 RMSE 预选 AE 模式，随后用完整 dropout40 从头重训该模式得到最终插补矩阵。训练 Notebook 根本**不读取 true_data.csv**，完整参考数据由第二份单独评价 Notebook 才读取。

**重要**：人工非零遮盖不能提供真实零负标签，所以验证不能证明误补低；同时本流程属于同一数据集的独立目标位置验证，并非跨数据集迁移测试。

使用 `RUN_MODE='full'` 和 `V38_MODE='full'` 完整训练（默认）；smoke 仅供调试。GPU/CPU 训练耗时取决于机器。本版不加 GDCC / NAF / GeneCoRead / V35-V36 识别器。


## 原始读取、工具函数与数据预处理（原样保留）


In [ ]:
# 运行环境预检查：缺少依赖时在这里给出明确提示。
import importlib.util, sys, os
_required = ['numpy', 'pandas', 'scipy', 'sklearn', 'torch', 'torch_geometric', 'scanpy']
_missing = [name for name in _required if importlib.util.find_spec(name) is None]
if _missing:
    raise ImportError(
        '缺少依赖: ' + ', '.join(_missing) + '\n'
        '请在当前 Notebook 所用的 Python 环境中安装缺失包；torch_geometric 需按本机 PyTorch/CUDA 版本选择对应安装命令。'
    )
print('依赖检查通过。Python:', sys.version.split()[0], '| PyTorch:', __import__('torch').__version__)


### 导库

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.neighbors import NearestNeighbors

from torch_geometric.nn import GCNConv
import pandas as pd
from sklearn.metrics import pairwise_distances
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import contingency_matrix
import warnings
import sys
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
# for cosine similarity metric used in evaluation
from sklearn.metrics.pairwise import cosine_similarity
import scanpy as sc
from functools import partial
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import math
import random
import copy
from sklearn.neighbors import NearestNeighbors
from torch_geometric.nn import GATConv

### 工具函数

In [ ]:
#加载数据
def load_data(data_path):
    data_csv = pd.read_csv(data_path, index_col=0)
    cells = data_csv.columns.values
    genes = data_csv.index.values
    data = data_csv.values.T


    return data, cells, genes

In [ ]:
#数据处理
def take_norm(data, cellwise_norm=True, log1p=True):
    data_norm = data.copy()
    data_norm = data_norm.astype('float32')
    if cellwise_norm:
        libs = data.sum(axis=1)
        norm_factor = np.diag(np.median(libs) / libs)
        data_norm = np.dot(norm_factor, data_norm)

    if log1p:
        data_norm = np.log2(data_norm + 1.)
    return data_norm

In [ ]:
#模拟dropout
def impute_dropout(X, seed=None, drop_rate=0.1):
    """
    X: original testing set
    ========
    returns:
    X_zero: copy of X with zeros
    """

    X_zero = np.copy(X)
    i, j = np.nonzero(X_zero)
    if seed is not None:
        np.random.seed(seed)

    ix = np.random.choice(range(len(i)), int(
        np.floor(drop_rate * len(i))), replace=False)
    X_zero[i[ix], j[ix]] = 0.0

    return X_zero

In [ ]:
def purity_score(y_true, y_pred):
    # compute contingency matrix (also called confusion matrix)
    contingency_matrix1 = contingency_matrix(y_true, y_pred)
    # return purity
    return np.sum(np.amax(contingency_matrix1, axis=0)) / np.sum(contingency_matrix1) 
def JaccardInd(ytrue,ypred):
    n = len(ytrue)
    a,b,c,d = 0,0,0,0
    for i in range(n-1):
        for j in range(i+1,n):
            if ((ypred[i] == ypred[j])&(ytrue[i]==ytrue[j])):
                a = a + 1
            elif ((ypred[i] == ypred[j])&(ytrue[i]!=ytrue[j])):
                b = b + 1
            elif ((ypred[i] != ypred[j])&(ytrue[i]==ytrue[j])):
                c = c + 1
            else:
                d = d + 1
    if (a==0)&(b==0)&(c==0):
        return 0
    else:
        return a/(a+b+c)

In [ ]:
#评估指标
def cos_simility(imputed_data, original_data):
    # 使用 sklearn 的 cosine_similarity
    return np.mean(cosine_similarity(original_data, imputed_data))

def l1_distance(imputed_data, original_data):

    return np.mean(np.abs(original_data-imputed_data))

def RMSE(imputed_data, original_data):
    return np.sqrt(np.mean((original_data - imputed_data)**2))
 # 计算两个向量person相关系数
def pearson_corr(imputed_data, original_data):
    Y = original_data
    fake_Y = imputed_data
    fake_Y, Y = fake_Y.reshape(-1), Y.reshape(-1)
    fake_Y_mean, Y_mean = np.mean(fake_Y), np.mean(Y)
    corr = (np.sum((fake_Y - fake_Y_mean) * (Y - Y_mean))) / (
            np.sqrt(np.sum((fake_Y - fake_Y_mean) ** 2)) * np.sqrt(np.sum((Y - Y_mean) ** 2)))
    return corr

### 输入 dropout40（训练阶段不读取真实完整矩阵）

In [ ]:
#定义数据集名称和dropout率
# dataset_name = 'Zeisel'
# drop_rate = 0.4
device='cpu'

In [ ]:
# 与原数据读取函数 load_data() 保持一致；但训练阶段物理隔离 true_data.csv。
from pathlib import Path
_DATA_DIR_CANDIDATES = [Path(r'E:/code/my/zeisel_data'), Path.cwd(), Path.cwd()/'zeisel_data', Path('/mnt/data')]
def _find_dataset_file(filename):
    for directory in _DATA_DIR_CANDIDATES:
        p=directory/filename
        if p.is_file():return p.resolve()
    raise FileNotFoundError(filename+' 未找到；请放在 E:/code/my/zeisel_data 或 Notebook 目录')
DROPOUT_PATH=_find_dataset_file('Zeisel_dropout_40_gene_cell.csv')
DATA_DIR=DROPOUT_PATH.parent
drop_data, cells, genes=load_data(str(DROPOUT_PATH))
print('V38 TRAIN ONLY dropout40:',DROPOUT_PATH,'shape',drop_data.shape)
# 注意：故意不定义 groundTruth_data、reference_counts、Y、TRUE_PATH。


In [ ]:
print('训练输入:',drop_data.shape,'细胞:',len(cells),'基因:',len(genes))

In [ ]:
# # 模拟dropout事件
# drop_data = impute_dropout(groundTruth_data, drop_rate=drop_rate)
# print('dataset: {}, drop rate: {}'.format(dataset_name, drop_rate))

In [ ]:
# 训练中仅保留稀疏输入 X，不创建真实参考目标 Y。
X=torch.FloatTensor(np.copy(drop_data))

In [ ]:
# 保留原数据处理函数，但所有 V38 训练严格使用 X_count（原始 count），不使用 log1p 数据。
X_norm=take_norm(drop_data,cellwise_norm=True,log1p=True)
X_input=torch.FloatTensor(X_norm).to(device)
n_cells,n_genes=X_input.shape
print('cells:',n_cells,'genes:',n_genes)


## 公共 count 输入与原构图配置


In [ ]:
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from IPython.display import display
from contextlib import contextmanager
import hashlib
import gc
import math
import random
import time

# 保留原构图与 OwnOnly 配置；GAT 的训练入口保留 v31_train；仅启用 GAT_Residual 分支。
CONFIG = dict(
    seed=42, deterministic=False, model_device='auto', cpu_threads=1,
    ae_epochs=300, ae_lr=1e-3, ae_hidden=512, ae_latent=128,
    ae_dropout=0.1, ae_batch_size=256, ae_mask_ratio=0.15,
    cl_weight=0.1, cl_temperature=0.2, cl_warmup=50, cl_ramp_epochs=50,
    graph_k=15, selfexpr_l1=0.05, selfexpr_l2=0.01,
    selfexpr_max_steps=200, selfexpr_min_steps=50,
    selfexpr_tol=1e-5, selfexpr_edge_epsilon=1e-6,
    branch_epochs=120, fusion_epochs=60, branch_lr=1e-3, fusion_lr=1e-3,
    loss='mse', feature_dim=64, self_hidden=128, gat_heads=4,
    attention_dropout=0.1, sk_hidden=32, mask_ratio=0.15,
    gate_bound=0.1, gate_anchor_weight=0.02,
    weak_zero_weight=0.05, weak_zero_sample_ratio=0.5,
    pair_block=32768, cell_block=256, gene_block=512,
    grad_clip=5.0, log_every=10,
)
RUN_MODE = 'full'  # 一键正式运行，不需要手动切换模式。
if RUN_MODE == 'smoke':
    CONFIG.update(ae_epochs=3, cl_warmup=0, cl_ramp_epochs=1,
                  selfexpr_max_steps=3, selfexpr_min_steps=2,
                  branch_epochs=2, fusion_epochs=2)
elif RUN_MODE != 'full':
    raise ValueError("RUN_MODE 必须是 full 或 smoke")

ENCODER_VARIANTS = (
    dict(name='OneView_AE_Count', views=1),
    dict(name='TwoView_SharedAE_Count_V26Graph', views=2),
)

def expression_digest(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256(str((a.shape, str(a.dtype))).encode())
    h.update(a.tobytes())
    return h.hexdigest()

def graph_digest(graph):
    w = graph.tocsr(copy=True); w.sort_indices()
    h = hashlib.sha256(str(w.shape).encode())
    for a in (w.indptr, w.indices, w.data):
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()

@contextmanager
def controlled_rng(seed, device):
    # 同一个主 seed 派生的固定随机流。退出时恢复外部 RNG，绝不是多 seed 实验。
    dev = torch.device(device)
    devices = [dev.index if dev.index is not None else torch.cuda.current_device()] if dev.type == 'cuda' else []
    with torch.random.fork_rng(devices=devices):
        torch.random.default_generator.manual_seed(int(seed))
        if dev.type == 'cuda':
            with torch.cuda.device(dev):
                torch.cuda.manual_seed(int(seed))
        yield

@contextmanager
def comparison_cpu_scope(device, cfg):
    old = torch.get_num_threads()
    if torch.device(device).type == 'cpu':
        torch.set_num_threads(int(cfg['cpu_threads']))
    try:
        yield
    finally:
        torch.set_num_threads(old)

torch.use_deterministic_algorithms(bool(CONFIG['deterministic']))
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
model_device = torch.device(('cuda' if torch.cuda.is_available() else 'cpu')
                            if CONFIG['model_device'] == 'auto' else CONFIG['model_device'])
count_array = np.asarray(drop_data)
if count_array.ndim != 2 or min(count_array.shape) < 2:
    raise ValueError('dropout count 必须为细胞×基因矩阵，至少两个细胞和两个基因')
if not np.isfinite(count_array).all():
    raise ValueError('count 输入包含 NaN/Inf')
if (count_array < 0).any():
    raise ValueError('输入必须非负')
if not (count_array > 0).any() or (count_array > 0).all():
    raise ValueError('需要同时存在观测非零和未知零')
X_count = torch.as_tensor(np.ascontiguousarray(count_array, dtype=np.float32), device=model_device)
X_graph = X_count  # 明确使用 count；不读取原预处理单元生成的 X_norm / X_input。
if not bool(torch.isfinite(X_count).all()):
    raise ValueError('count 超出 float32 范围')
_v20_input_hash = expression_digest(count_array)
_v20_tensor_hash = expression_digest(X_count.detach().cpu().numpy())
print('V38：AE 重构目标 A/B/C 比较；仅使用 dropout40。')
print('主 seed=42 | 设备:', model_device, '| 细胞×基因:', count_array.shape, '| 模式:', RUN_MODE)
print('本版不自动重跑单视图或完整 V14。')
print('构图 AE 输入/重构目标均为原始 count，不使用 X_norm，也不对 count 做 log 或库大小归一化。')

## 公共工具与 OwnOnly 初始化

历史 SK 初始化类仅用于保持 OwnOnly 的初始化随机数顺序，不执行 SK 前向或训练。


In [ ]:
import gc


import math


import random


import time


import numpy as np


import pandas as pd


from scipy import sparse


import torch


from torch import nn


from torch.nn import functional as F


from torch_geometric.nn import GATv2Conv


def sync_device(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)


def slices(size, block):
    for start in range(0, size, int(block)):
        yield slice(start, min(size, start+int(block)))


def seed_recovery(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def prepare_recovery_graph(graph, raw, device):
    """W[i,j] is a message from j to i. Graph/evidence use only dropout input."""
    w = graph.tocsr(copy=True).astype(np.float32)
    if w.shape != (raw.shape[0], raw.shape[0]):
        raise ValueError('图的细胞数与原dropout输入不一致')
    if w.diagonal().any():
        w.setdiag(0)
    w.eliminate_zeros(); w.sort_indices()
    if not np.isfinite(w.data).all() or (w.data < 0).any():
        raise ValueError('SelfExpr权重必须有限且非负')
    sums = np.asarray(w.sum(axis=1)).ravel()
    inv = np.divide(1., sums, out=np.zeros_like(sums), where=sums > 0)
    w = (sparse.diags(inv) @ w).tocsr()
    coo = w.tocoo()
    ij = torch.as_tensor(np.stack([coo.row, coo.col]), dtype=torch.long, device=device)
    values = torch.as_tensor(coo.data, device=device)
    graph_t = dict(
        W=torch.sparse_coo_tensor(ij, values, w.shape, device=device).coalesce(),
        edge_index=torch.stack([ij[1], ij[0]]),
        edge_attr=torch.log(values.clamp_min(1e-8))[:, None],
        has_neighbor=torch.as_tensor(np.asarray(w.getnnz(axis=1)).ravel() > 0, device=device),
    )
    return graph_t


@torch.no_grad()
def recovery_evidence(corrupted, graph, gene_block=512):
    """Recompute from masked input. Hidden values never enter neighbor statistics."""
    support = torch.empty_like(corrupted)
    disagreement = torch.empty_like(corrupted)
    for gs in slices(corrupted.shape[1], gene_block):
        values = corrupted[:, gs]
        nonzero = (values > 0).to(values.dtype)
        log_values = torch.log1p(values)
        p = torch.sparse.mm(graph['W'], nonzero).clamp(0., 1.)
        denominator = p.clamp_min(1e-8)
        mean = torch.sparse.mm(graph['W'], log_values)/denominator
        second = torch.sparse.mm(graph['W'], log_values.square())/denominator
        sd = (second-mean.square()).clamp_min(0.).sqrt()
        sd = torch.where(p > 1e-8, sd, torch.zeros_like(sd))
        support[:, gs] = p
        disagreement[:, gs] = sd/(1.+sd)
    return dict(support=support, disagreement=disagreement)


def sample_recovery_task(raw, zero_pool, cfg):
    """Fresh mask of observed nonzeros; untouched input zeros remain unlabeled."""
    observed = raw > 0
    hidden = observed & (torch.rand_like(raw) < float(cfg['mask_ratio']))
    if not bool(hidden.any()):
        # 小型数据的兜底；仍只遮盖一个确实观测非零的位置。
        first = observed.nonzero(as_tuple=False)[0]
        hidden[first[0], first[1]] = True
    positive = hidden.nonzero(as_tuple=False)
    corrupted = raw.masked_fill(hidden, 0.)
    negative_count = min(zero_pool.numel(), int(math.ceil(len(positive)*cfg['weak_zero_sample_ratio'])))
    if negative_count and cfg['weak_zero_weight'] > 0.:
        # 均匀有放回抽样，避免每轮对整个零证据池进行昂贵的randperm。
        chosen = zero_pool[torch.randint(zero_pool.numel(), (negative_count,), device=raw.device)]
        negative = torch.stack([chosen//raw.shape[1], chosen%raw.shape[1]], dim=1)
    else:
        negative = torch.empty((0,2), dtype=torch.long, device=raw.device)
    pairs = torch.cat([positive, negative], dim=0)
    targets = torch.cat([raw[positive[:,0], positive[:,1]], raw.new_zeros(len(negative))])
    # positive/weak zero分开取均值。总目标不是把所有原始零都当0重构。
    weights = torch.cat([
        raw.new_full((len(positive),), 1./len(positive)),
        raw.new_full((len(negative),), cfg['weak_zero_weight']/max(1,len(negative))),
    ])
    return dict(corrupted=corrupted, hidden=hidden, pairs=pairs, targets=targets,
                weights=weights, positive_count=len(positive), weak_zero_count=len(negative))


class RecoverySKGate(nn.Module):
    """32-dim gate; no observed flag; initialized to exactly equal branch weights."""
    def __init__(self, feature_dim, genes, hidden=32, bound=.1):
        super().__init__()
        self.bound = float(bound)
        self.local = nn.Linear(6, hidden)
        self.context = nn.Linear(feature_dim, hidden, bias=False)
        self.select = nn.Linear(hidden, 2)
        self.gene_bias = nn.Parameter(torch.zeros(genes,2))
        nn.init.zeros_(self.select.weight); nn.init.zeros_(self.select.bias)

    def forward(self, own_mu, nbr_mu, own_feature, scale, support, disagreement,
                gene_bias, has_neighbor, cell_context=None):
        own_log = torch.log1p(own_mu/scale)
        nbr_log = torch.log1p(nbr_mu/scale)
        local = torch.stack([own_log, nbr_log, own_log-nbr_log,
                             (own_log-nbr_log).abs(), support, disagreement], dim=-1)
        context = self.context(own_feature) if cell_context is None else cell_context
        if local.ndim == 3 and context.ndim == 2:
            context = context[:,None,:]
        hidden = F.relu(self.local(local)+context)
        weights = (self.select(hidden)+gene_bias).softmax(dim=-1)
        nbr_weight = self.bound+(1.-2.*self.bound)*weights[...,1]
        nbr_weight = torch.where(has_neighbor, nbr_weight, torch.zeros_like(nbr_weight))
        return (1.-nbr_weight)*own_mu+nbr_weight*nbr_mu, nbr_weight


class RecoverySKImputer(nn.Module):
    def __init__(self, raw, cfg):
        super().__init__()
        genes, dim = raw.shape[1], int(cfg['feature_dim'])
        self.encoder = nn.Sequential(
            nn.Linear(genes,int(cfg['self_hidden'])), nn.LayerNorm(int(cfg['self_hidden'])),
            nn.ReLU(), nn.Linear(int(cfg['self_hidden']),dim), nn.LayerNorm(dim), nn.ReLU())
        self.gat = GATv2Conv(dim,dim//cfg['gat_heads'],heads=cfg['gat_heads'],concat=True,
                             dropout=cfg['attention_dropout'],add_self_loops=False,edge_dim=1,bias=False)
        self.neighbor_norm = nn.LayerNorm(dim)
        self.self_head = nn.Linear(dim,genes)
        self.neighbor_head = nn.Linear(dim,genes)
        self.fusion = RecoverySKGate(dim,genes,cfg['sk_hidden'],cfg['gate_bound'])
        self.register_buffer('gene_scale', raw.detach().mean(dim=0).clamp_min(.1))
        for head in (self.self_head,self.neighbor_head):
            nn.init.normal_(head.weight,std=.01)
            nn.init.constant_(head.bias, math.log(math.expm1(1.)))

    def encode(self, corrupted, graph):
        own = self.encoder(torch.log1p(corrupted))
        nbr = F.elu(self.neighbor_norm(self.gat(own,graph['edge_index'],graph['edge_attr'])))
        return own,nbr

    def branch_pairs(self, own, nbr, pairs, graph):
        ci,gi = pairs[:,0],pairs[:,1]
        scale = self.gene_scale[gi]
        sm = F.softplus((own[ci]*self.self_head.weight[gi]).sum(dim=-1)+self.self_head.bias[gi])*scale+1e-5
        nm = F.softplus((nbr[ci]*self.neighbor_head.weight[gi]).sum(dim=-1)+self.neighbor_head.bias[gi])*scale+1e-5
        nm = torch.where(graph['has_neighbor'][ci],nm,sm)
        return sm,nm

    def fusion_pairs(self, own, sm, nm, pairs, evidence, graph, cell_context=None):
        ci,gi = pairs[:,0],pairs[:,1]
        return self.fusion(sm,nm,own[ci],self.gene_scale[gi],
                           evidence['support'][ci,gi],evidence['disagreement'][ci,gi],
                           self.fusion.gene_bias[gi],graph['has_neighbor'][ci],
                           None if cell_context is None else cell_context[ci])


def preserve_observed(prediction, raw):
    # 保持 raw 的原始精度；只替换 dropout 输入的零位置。
    x = np.asarray(raw)
    result = np.asarray(prediction, dtype=np.float64).copy()
    result[x > 0] = x[x > 0]
    if not np.array_equal(result[x > 0], x[x > 0]):
        raise AssertionError('观测值未保留')
    return result


def pearson_flat(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size < 2 or a.size != b.size:
        return np.nan
    ac, bc = a-a.mean(), b-b.mean()
    den = np.sqrt(np.dot(ac, ac))*np.sqrt(np.dot(bc, bc))
    return float(np.dot(ac, bc)/den) if den > 0 else np.nan


def metric_values(prediction, truth, mask=None):
    if mask is None:
        p, y = np.asarray(prediction).ravel(), np.asarray(truth).ravel()
    else:
        p, y = np.asarray(prediction)[mask], np.asarray(truth)[mask]
    p, y = p.astype(np.float64), y.astype(np.float64)
    if not p.size:
        return dict(n=0, PCC=np.nan, MSE=np.nan, RMSE=np.nan, L1=np.nan,
                    bias=np.nan, pred_mean=np.nan, truth_mean=np.nan)
    error = p-y
    mse = float(np.mean(error*error))
    return dict(n=int(p.size), PCC=pearson_flat(p, y), MSE=mse, RMSE=float(np.sqrt(mse)),
                L1=float(np.abs(error).mean()), bias=float(error.mean()),
                pred_mean=float(p.mean()), truth_mean=float(y.mean()))


def frozen_model_hash(model):
    """Stable digest used to verify that the frozen own predictor is unchanged."""
    import hashlib
    h = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        h.update(name.encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()



## 原 SelfExpr 构图


In [ ]:
def row_normalize(a):
    a=a.tocsr(copy=True).astype(np.float32)
    a.eliminate_zeros();a.sort_indices()
    if not np.isfinite(a.data).all() or (a.data<0).any():
        raise ValueError('图权重必须非负且有限')
    sums=np.asarray(a.sum(axis=1)).ravel()
    inv=np.divide(1.,sums,out=np.zeros_like(sums),where=sums>0)
    return (sparse.diags(inv)@a).tocsr()


def topk_matrix(a,k,include_self=False):
    """仅筛选矩阵元素；行循环不是细胞/基因分批训练。并列权重按列号稳定排序。"""
    n=a.shape[0]
    k=min(int(k),n if include_self else n-1)
    if k<1:raise ValueError('邻居数必须为正')
    is_sparse=sparse.issparse(a)
    if is_sparse:a=a.tocsr(copy=True);a.eliminate_zeros()
    rows=[];cols=[];values=[]
    for i in range(n):
        if is_sparse:
            start,stop=a.indptr[i:i+2];j=a.indices[start:stop];v=a.data[start:stop]
        else:
            j=np.arange(n);v=np.asarray(a[i])
        keep=(v>0)&np.isfinite(v)
        if not include_self:keep&=(j!=i)
        j=j[keep];v=v[keep]
        chosen=np.lexsort((j,-v))[:k]
        rows.extend([i]*len(chosen));cols.extend(j[chosen]);values.extend(v[chosen])
    return sparse.csr_matrix((values,(rows,cols)),shape=(n,n),dtype=np.float32)


def mutual_topk(a,k):
    directed=topk_matrix(a,k)
    topology=directed.copy();topology.data[:]=1
    mutual=topology.multiply(topology.T)
    symmetric=((directed+directed.T)*.5).multiply(mutual).tocsr()
    return row_normalize(symmetric)


def graph_stats(w):
    degree=np.asarray(w.getnnz(axis=1)).ravel()
    components,_=connected_components(w,directed=False)
    return dict(edges=w.nnz//2,mean_neighbors=float(degree.mean()),
                max_neighbors=int(degree.max()),isolated=int((degree==0).sum()),
                components=int(components))


@torch.no_grad()
def learn_self_expression(processed,valid,cfg):
    """全细胞候选的非负稀疏自表达，整矩阵近端加速梯度；无 KNN 预筛选。"""
    if processed.ndim!=2:raise ValueError('processed 必须为细胞×特征矩阵')
    n,g=processed.shape
    l1=float(cfg['selfexpr_l1']);l2=float(cfg['selfexpr_l2'])
    if l1<0 or l2<0:raise ValueError('正则强度不能为负')
    if int(cfg['selfexpr_max_steps'])<1:raise ValueError('selfexpr_max_steps 必须为正')
    allowed=torch.as_tensor(valid,dtype=processed.dtype,device=processed.device)
    C=torch.zeros((n,n),dtype=processed.dtype,device=processed.device)
    extrapolated=C.clone()
    momentum=1.0
    # ||Z||_F²/G >= ||Z||_2²/G，使用保守的 Lipschitz 上界，避免估计误差。
    lipschitz=float(processed.square().sum().item()/g)+l2
    if lipschitz<=0:raise ValueError('构图矩阵不能全为零')
    step=1.0/lipschitz
    rows=[];last_logged=0
    for iteration in range(1,int(cfg['selfexpr_max_steps'])+1):
        residual=extrapolated@processed-processed
        gradient=(residual@processed.T)/g+l2*extrapolated
        # 非负 L1 近端算子；diag(C)=0，禁止细胞直接复制自己。
        new_C=(extrapolated-step*gradient-step*l1).clamp_min_(0)
        new_C.fill_diagonal_(0)
        new_C.mul_(allowed[:,None]).mul_(allowed[None,:])
        delta=new_C-C
        relative_change=float((torch.linalg.vector_norm(delta)/
            torch.linalg.vector_norm(C).clamp_min(1e-8)).item())
        next_momentum=(1+math.sqrt(1+4*momentum*momentum))/2
        extrapolated=new_C+((momentum-1)/next_momentum)*delta
        C=new_C;momentum=next_momentum
        del residual,gradient,new_C,delta
        should_stop=iteration>=cfg['selfexpr_min_steps'] and relative_change<cfg['selfexpr_tol']
        if iteration==1 or iteration%cfg['log_every']==0 or iteration==cfg['selfexpr_max_steps'] or should_stop:
            error=C@processed-processed
            mse=float(error.square().mean().item())
            objective=0.5*mse+l1*float(C.sum().item())/n+0.5*l2*float(C.square().sum().item())/n
            fraction=float((C>cfg['selfexpr_edge_epsilon']).float().sum().item()/max(n*(n-1),1))
            rows.append(dict(step=iteration,feature_reconstruction_MSE=mse,
                             objective_per_cell=objective,relative_change=relative_change,
                             nonzero_coefficient_fraction=fraction))
            print('稀疏自表达 %d/%d | feature MSE %.6f | objective %.6f | 相对变化 %.3g'%(
                iteration,cfg['selfexpr_max_steps'],mse,objective,relative_change))
            last_logged=iteration
            del error
        if should_stop:break
    converged=relative_change<cfg['selfexpr_tol'] and iteration>=cfg['selfexpr_min_steps']
    print('自表达停止:',('达到相对变化阈值' if converged else '达到迭代上限，未达到变化阈值'))
    # 先对称化，再仅保留最强 k 项；不把 N×N 的稠密系数全部转换为稀疏边。
    symmetric=(C+C.T)*0.5
    k=min(int(cfg['graph_k']),n-1)
    if k<1:raise ValueError('graph_k 必须为正')
    values,cols=torch.topk(symmetric,k=k,dim=1)
    rows_tensor=torch.arange(n,device=processed.device)[:,None].expand_as(cols)
    keep=values>cfg['selfexpr_edge_epsilon']
    affinity=sparse.csr_matrix((values[keep].cpu().numpy(),
        (rows_tensor[keep].cpu().numpy(),cols[keep].cpu().numpy())),shape=(n,n))
    graph=mutual_topk(affinity,cfg['graph_k'])
    history=pd.DataFrame(rows)
    return graph,history,dict(iterations=iteration,converged=converged,
                             l1=l1,l2=l2,step_size=step),None

## 双视图共享 AE：维持原构图方式


In [ ]:
class CountGraphAE(nn.Module):
    """原始 count 直接编码；预测同尺度非负 count，浅层用于 SelfExpr。"""
    def __init__(self, raw, cfg):
        super().__init__()
        genes = raw.shape[1]
        h, d = int(cfg['ae_hidden']), int(cfg['ae_latent'])
        self.shallow = nn.Sequential(nn.Linear(genes, h), nn.LayerNorm(h), nn.ReLU())
        self.deep = nn.Sequential(nn.Dropout(float(cfg['ae_dropout'])), nn.Linear(h, d))
        self.decoder = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, genes))
        # 只对预测做参数化：初始输出接近该基因 count 均值；输入和 MSE 目标不缩放。
        self.register_buffer('gene_scale', raw.detach().mean(dim=0).clamp_min(.1))
        nn.init.normal_(self.decoder[-1].weight, std=.01)
        nn.init.constant_(self.decoder[-1].bias, math.log(math.expm1(1.)))

    def forward(self, x):
        h = self.shallow(x)
        return F.softplus(self.decoder(self.deep(h)))*self.gene_scale, h


def shallow_contrast_loss(h1, h2, temperature):
    """同细胞跨视图为正；其他细胞的同视图与跨视图表示均为负。"""
    if h1.ndim != 2 or h1.shape != h2.shape or len(h1) < 2:
        raise ValueError('两路特征须同形，且至少包含两个有效细胞')
    if temperature <= 0:
        raise ValueError('temperature 必须大于 0')
    n = h1.shape[0]
    a = F.normalize(h1, dim=1, eps=1e-8)
    b = F.normalize(h2, dim=1, eps=1e-8)
    z = torch.cat([a, b], dim=0)
    similarity = z @ z.T
    self_mask = torch.eye(2*n, dtype=torch.bool, device=z.device)
    logits = (similarity / float(temperature)).masked_fill(self_mask, float('-inf'))
    anchors = torch.arange(2*n, device=z.device, dtype=torch.long)
    positive_ids = (anchors+n) % (2*n)
    loss = F.cross_entropy(logits, positive_ids)
    with torch.no_grad():
        positive = similarity[anchors, positive_ids].mean()
        negative_mask = ~self_mask
        negative_mask[anchors, positive_ids] = False
        negative = similarity[negative_mask].mean()
        accuracy = (logits.argmax(dim=1) == positive_ids).float().mean()
    return loss, dict(positive_cosine=float(positive), negative_cosine=float(negative),
                      pair_accuracy=float(accuracy))


def contrast_batches(order, block):
    # 最后一批若只有一个细胞，合并到前一批，避免没有负样本。
    parts = list(order.split(int(block)))
    if len(parts) > 1 and len(parts[-1]) == 1:
        parts[-2] = torch.cat([parts[-2], parts[-1]])
        parts.pop()
    return parts


def make_ae_views(target, cfg):
    # target 为原始 count；只临时遮盖观测非零，不改变原输入或进行归一化。
    observed = target > 0
    mask1 = observed & (torch.rand_like(target) < float(cfg['ae_mask_ratio']))
    mask2 = observed & (torch.rand_like(target) < float(cfg['ae_mask_ratio']))
    return target.masked_fill(mask1, 0.), target.masked_fill(mask2, 0.)


@torch.no_grad()
def shallow_blocks(models, x, cfg):
    blocks = []
    for model in models:
        model.eval()
        blocks.append(torch.cat([model.shallow(x[cs]) for cs in slices(len(x), cfg['ae_batch_size'])]))
    return blocks


@torch.no_grad()
def fit_shallow_graph_transform(blocks, valid):
    # 学到的浅层特征做中心化和整块 RMS 缩放，控制 SelfExpr 正则的尺度。
    # 这一步作用于隐藏特征，不是对输入 count 做库大小归一化或 log。
    transforms, arrays, diagnostics = [], [], []
    for index, h in enumerate(blocks):
        center = h[valid].mean(dim=0, keepdim=True)
        energy = (h[valid] - center).square().mean()
        if not bool(torch.isfinite(energy)) or float(energy) < 1e-12:
            raise FloatingPointError('浅层特征接近塌缩，停止构图')
        scale = energy.sqrt()
        value = (h-center)/scale
        value[~valid] = 0.
        arrays.append(value)
        transforms.append(dict(center=center.detach(), scale=scale.detach()))
        diagnostics.append(dict(encoder=index+1, width=h.shape[1],
                                centered_MSE=float(energy),
                                mean_feature_std=float(h[valid].std(dim=0, unbiased=False).mean())))
    return torch.cat(arrays, dim=1).detach(), transforms, pd.DataFrame(diagnostics)


def validate_encoder_cfg(cfg):
    for key in ('ae_epochs', 'ae_hidden', 'ae_latent', 'ae_batch_size', 'cl_ramp_epochs', 'cpu_threads'):
        if int(cfg[key]) < 1:
            raise ValueError(key + ' 必须为正整数')
    if cfg['ae_batch_size'] < 2 or cfg['ae_hidden'] < 2:
        raise ValueError('ae_batch_size 和 ae_hidden 至少为 2')
    if not 0. <= cfg['ae_mask_ratio'] < 1. or not 0. <= cfg['ae_dropout'] < 1.:
        raise ValueError('AE 遮盖率和隐藏层 dropout 必须在 [0,1)')
    if cfg['cl_weight'] < 0. or cfg['cl_temperature'] <= 0. or cfg['cl_warmup'] < 0:
        raise ValueError('对比权重/温度/预热配置无效')


def v38_ae_reconstruction(prediction, target, masked_view, cfg):
    """AE reconstruction WITHOUT interpreting original unknown zeros as observed negatives in B/C.
    No reference/ground truth is available. Both masks and targets derived from current dropout40 fit input.
    """
    mode=cfg.get('ae_loss_mode','A_full')
    if mode=='A_full':
        return F.mse_loss(prediction,target)
    obs=target>0
    masked=obs & (masked_view==0)
    keep=obs & (masked_view>0)
    if not bool(masked.any()):
        # Rare degenerate tiny batch; do not treat original zero as a negative.
        if not bool(obs.any()):
            return prediction.sum()*0.
        masked=obs
    recovery=(prediction[masked]-target[masked]).square().mean()
    if mode=='B_masked':
        return recovery
    if mode=='C_masked_observed':
        observed=(prediction[keep]-target[keep]).square().mean() if bool(keep.any()) else prediction.sum()*0.
        return recovery+float(cfg.get('ae_observed_weight',0.15))*observed
    raise ValueError('Unknown V38 AE loss mode: '+str(mode))


def train_shallow_encoders(x, spec, cfg):
    validate_encoder_cfg(cfg)
    if spec['views'] not in (1, 2):
        raise ValueError('只支持单视图 AE 与双视图共享 AE')
    valid = (x > 0).any(dim=1)
    indices = torch.arange(len(x), device=x.device, dtype=torch.long)
    if int(valid.sum()) < 2:
        raise ValueError('至少需要两个有效细胞参与构图比较')
    with controlled_rng(cfg['seed'], x.device):
        models = nn.ModuleList([CountGraphAE(x, cfg).to(x.device)])
    # 每组只有一套 AE；双视图前向调用同一对象，编码器/解码器全部共享。
    model = models[0]
    initial_hashes = tuple(frozen_model_hash(m) for m in models)
    # 原始 count 全位置 MSE 除以同一固定标量，避免数值尺度淹没对比项。
    # 没有逐基因/逐位置加权，也没有将 count 目标换成 log 或 norm。
    mse_scale = float((x.double().square().mean() if cfg.get('ae_loss_mode','A_full')=='A_full' else x[x>0].double().square().mean()).clamp_min(1.).cpu())
    if not math.isfinite(mse_scale):
        raise FloatingPointError('count MSE 缩放标量非有限')
    optimizer = torch.optim.Adam(models.parameters(), lr=cfg['ae_lr'])
    history = []
    sync_device(x.device); started = time.perf_counter()
    for epoch in range(1, int(cfg['ae_epochs'])+1):
        models.train(); optimizer.zero_grad(set_to_none=True)
        ramp = min(1., max(0., (epoch-int(cfg['cl_warmup'])) / int(cfg['cl_ramp_epochs'])))
        weight = float(cfg['cl_weight']) * ramp if spec['views'] == 2 else 0.
        totals = dict(reconstruction_MSE=0., reconstruction_MSE_scaled=0., contrast_loss=0., total_loss=0.,
                      positive_cosine=0., negative_cosine=0., pair_accuracy=0.)
        sync_device(x.device); tick = time.perf_counter()
        # 每轮覆盖全部细胞和基因（含原始零），累积梯度后更新一次。
        with controlled_rng(int(cfg['seed'])*1000003+epoch, x.device):
            order = indices[torch.randperm(len(indices), device=x.device)]
            for ids in contrast_batches(order, cfg['ae_batch_size']):
                target = x[ids]
                # 分离随机流：额外视图随机数不会改变另一组的细胞顺序和首路隐藏 Dropout。
                batch_id = int(ids[0])
                base_stream = int(cfg['seed'])*1000003 + epoch*100000 + batch_id*10
                if spec['views'] == 1:
                    with controlled_rng(base_stream+2, x.device):
                        p1, h1 = model(target)
                    rec_raw = v38_ae_reconstruction(p1, target, target, cfg)
                    cl = rec_raw.new_zeros(())
                    diagnostics = dict(positive_cosine=0., negative_cosine=0., pair_accuracy=0.)
                else:
                    with controlled_rng(base_stream+1, x.device):
                        a, b = make_ae_views(target, cfg)
                    with controlled_rng(base_stream+2, x.device):
                        p1, h1 = model(a)
                    with controlled_rng(base_stream+3, x.device):
                        p2, h2 = model(b)
                    rec_raw = .5*(v38_ae_reconstruction(p1,target,a,cfg) + v38_ae_reconstruction(p2,target,b,cfg))
                    contrast_valid = (target > 0).any(dim=1)
                    if int(contrast_valid.sum()) >= 2:
                        cl, diagnostics = shallow_contrast_loss(
                            h1[contrast_valid], h2[contrast_valid], cfg['cl_temperature'])
                    else:
                        cl = rec_raw.new_zeros(())
                        diagnostics = dict(positive_cosine=0., negative_cosine=0., pair_accuracy=0.)
                rec = rec_raw/mse_scale
                loss = rec + weight*cl
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError('AE/对比损失非有限')
                fraction = len(ids)/len(indices)
                (loss*fraction).backward()
                for key, value in dict(reconstruction_MSE=float(rec_raw.detach()),
                                       reconstruction_MSE_scaled=float(rec.detach()), contrast_loss=float(cl.detach()),
                                       total_loss=float(loss.detach()), **diagnostics).items():
                    totals[key] += fraction*value
        torch.nn.utils.clip_grad_norm_(models.parameters(), cfg['grad_clip'], error_if_nonfinite=True)
        optimizer.step()
        sync_device(x.device); seconds = time.perf_counter()-tick
        history.append(dict(variant=spec['name'], seed=cfg['seed'], epoch=epoch,
                            views=spec['views'], MSE_scale=mse_scale,
                            contrast_weight=weight, epoch_seconds=seconds, **totals))
        if epoch == 1 or epoch % cfg['log_every'] == 0 or epoch == cfg['ae_epochs']:
            print(f'{spec["name"]} AE {epoch}/{cfg["ae_epochs"]} | count MSE {totals["reconstruction_MSE"]:.4f}'
                  f' | MSE/scale {totals["reconstruction_MSE_scaled"]:.6f}'
                  f' | CL {totals["contrast_loss"]:.4f} × {weight:.3f} | {seconds:.2f}s/轮')
    optimizer.zero_grad(set_to_none=True)
    models.eval().requires_grad_(False)
    features, transforms, feature_df = fit_shallow_graph_transform(shallow_blocks(models, x, cfg), valid)
    feature_df.insert(0, 'variant', spec['name'])
    meta = dict(variant=spec['name'], stage='count_graph_AE', epochs=cfg['ae_epochs'],
                encoders=1, views=spec['views'], MSE_scale=mse_scale,
                parameters=sum(p.numel() for p in models.parameters()),
                graph_feature_width=features.shape[1], train_seconds=time.perf_counter()-started,
                initial_encoder_hashes=initial_hashes)
    del optimizer
    return models, features, transforms, pd.DataFrame(history), feature_df, meta


def matched_recovery_task(raw, zero_pool, cfg, stream, epoch):
    # 保证不同图的边数、弱零抽样数和 GAT dropout 不改变下一轮的正例遮盖。
    with controlled_rng(int(cfg['seed'])*1000003 + int(stream)*10000 + int(epoch), raw.device):
        return sample_recovery_task(raw, zero_pool, cfg)


## SelfExpr 图接口


In [ ]:
def graph_for_context(features, valid, raw, cfg):
    if bool(valid.any()) and bool(features.square().sum() > 0):
        b, history, status, _ = learn_self_expression(features, valid.detach().cpu().numpy(), cfg)
    else:
        b = sparse.csr_matrix((len(raw), len(raw)), dtype=np.float32)
        history = pd.DataFrame()
        status = dict(iterations=0, converged=False)
    return b, prepare_recovery_graph(b, raw, raw.device), history, status



## 共享 OwnOnly 预测器


In [ ]:
import copy


def diagnostic_random_graph(matrix, seed):
    """逐行随机重选邻居：严格保留每行度数及边权集合，无自环/重复边。
    随机图不保证互近邻对称性，因此它是邻居身份破坏对照，不是完整拓扑匹配对照。
    """
    b = matrix.tocsr(copy=True).astype(np.float32)
    b.setdiag(0); b.eliminate_zeros(); b.sort_indices()
    rng = np.random.RandomState(seed)
    rr, cc, vv = [], [], []
    for i in range(b.shape[0]):
        lo, hi = b.indptr[i:i+2]; k = hi-lo
        choices = rng.choice(b.shape[0]-1, size=k, replace=False)
        choices += choices >= i
        rr.extend([i]*k); cc.extend(choices.tolist()); vv.extend(b.data[lo:hi].tolist())
    out = sparse.csr_matrix((vv,(rr,cc)),shape=b.shape,dtype=np.float32)
    assert np.array_equal(np.diff(out.indptr),np.diff(b.indptr))
    assert not out.diagonal().any()
    return out


class DiagnosticOwn(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.encoder=copy.deepcopy(base.encoder)
        self.head=copy.deepcopy(base.self_head)
        self.register_buffer('gene_scale',base.gene_scale.detach().clone())

    def encode(self,x):
        return self.encoder(torch.log1p(x))

    def pairs(self,h,pairs):
        ci,gi=pairs[:,0],pairs[:,1]
        return F.softplus((h[ci]*self.head.weight[gi]).sum(-1)+self.head.bias[gi])*self.gene_scale[gi]+1e-5


@torch.no_grad()
def diagnostic_own_predict(model,raw,cfg):
    model.eval(); h=model.encode(raw)
    out=np.empty(tuple(raw.shape),np.float32)
    for gs in slices(raw.shape[1],cfg['gene_block']):
        out[:,gs]=(F.softplus(F.linear(h,model.head.weight[gs],model.head.bias[gs]))*model.gene_scale[gs]+1e-5).cpu().numpy()
    return out


def diagnostic_own_backward(model,task,cfg,scale):
    h=model.encode(task['corrupted']); leaf=h.detach().requires_grad_(True)
    total=0.
    for ps in slices(len(task['pairs']),cfg['pair_block']):
        pred=model.pairs(leaf,task['pairs'][ps])
        loss=(task['weights'][ps]*(pred-task['targets'][ps]).square()).sum()/scale
        loss.backward(); total+=float(loss.detach())
    h.backward(leaf.grad)
    return total



## 公共基线训练及评价函数


In [ ]:
"""V26 base utilities: build the two-view SelfExpr graph, train the frozen own-only predictor, and evaluate only after inference."""
import copy

def v23_choose_graph(raw, cfg, trained=None, variant='TwoView_SharedAE_Count_V26Graph'):
    """复用已核对的固定图；否则只训练一套双视图 AE 和 SelfExpr，不重跑 V14。"""
    if trained is not None:
        artifact = trained.get('artifacts', {}).get(variant)
        if artifact is None:
            raise ValueError('comparison_run 中缺少 ' + variant)
        b = artifact['graph'].copy()
        old_scale = artifact['base'].gene_scale.detach().to(raw.device)
        if b.shape != (len(raw), len(raw)) or not torch.equal(old_scale, raw.mean(0).clamp_min(.1)):
            raise ValueError('已有 comparison_run 与当前 dropout 输入不匹配，请重新构图')
        observed = raw.cpu().numpy() > 0
        reference = np.asarray(trained['predictions'][variant])
        if reference.shape != tuple(raw.shape) or not np.array_equal(reference.astype(np.float32)[observed], raw.cpu().numpy()[observed]):
            raise ValueError('已有预测的观测值与当前输入不匹配')
        print('复用当前双视图 SelfExpr 图；不重训 AE、SelfExpr 或 GAT/SK。')
        return b, dict(source='existing_comparison_run', graph_hash=graph_digest(b))
    print('无已有图：仅训练双视图共享 AE 512/128 + SelfExpr；不训练单视图和 V14 后端。')
    spec = dict(name=variant, views=2)
    encoders, features, transforms, history, feature_df, meta = train_shallow_encoders(raw, spec, cfg)
    b, graph, graph_history, status = graph_for_context(features, (raw > 0).any(1), raw, cfg)
    del encoders, features, transforms, graph
    return b, dict(source='fresh_two_view', graph_hash=graph_digest(b),
                   ae_history=history, ae_meta=meta, graph_history=graph_history, graph_status=status)


def v23_train_own(raw, real_graph, random_graph, cfg):
    """复现 V22 独立 OwnOnly 的初始化、弱零池、遮盖随机流和训练轮数。
    V22 没有返回自身模型对象，只返回了预测，因此需要重新训练这一支。
    RecoverySKImputer 只用于匹配旧初始化随机数；未执行 GAT/SK 前向或优化。
    """
    ev1 = recovery_evidence(raw, real_graph, cfg['gene_block'])
    ev2 = recovery_evidence(raw, random_graph, cfg['gene_block'])
    common = ((raw == 0) & (ev1['support'] == 0) & (ev2['support'] == 0)
              & real_graph['has_neighbor'][:, None] & random_graph['has_neighbor'][:, None])
    pool = common.flatten().nonzero(as_tuple=False).flatten()
    del ev1, ev2, common
    seed_recovery(int(cfg['seed']) + 500)
    init = RecoverySKImputer(raw, cfg).to(raw.device)
    own = DiagnosticOwn(init).to(raw.device)
    del init
    scale = float(raw[raw > 0].double().square().mean().clamp_min(1.))
    opt = torch.optim.Adam(own.parameters(), lr=cfg['branch_lr'])
    rows = []
    for epoch in range(1, int(cfg['branch_epochs']) + 1):
        tick = time.perf_counter()
        task = matched_recovery_task(raw, pool, cfg, 2201, epoch)
        own.train(); opt.zero_grad(set_to_none=True)
        with controlled_rng(int(cfg['seed']) + epoch, raw.device):
            loss = diagnostic_own_backward(own, task, cfg, scale)
        torch.nn.utils.clip_grad_norm_(own.parameters(), cfg['grad_clip'], error_if_nonfinite=True)
        opt.step()
        sync_device(raw.device)
        rows.append(dict(stage='own', epoch=epoch, loss=loss, seconds=time.perf_counter()-tick))
        if epoch == 1 or epoch % cfg['log_every'] == 0 or epoch == cfg['branch_epochs']:
            print(f'OwnOnly {epoch}/{cfg["branch_epochs"]} | MSE/scale {loss:.6f} | {rows[-1]["seconds"]:.2f}s')
    opt.zero_grad(set_to_none=True)
    own.eval().requires_grad_(False)
    return own, scale, pd.DataFrame(rows), int(len(pool))


def v23_evaluate(raw_array, truth_array, result):
    """Only post-training evaluation: never select epochs, hyperparameters or output by truth."""
    x, y = np.asarray(raw_array), np.asarray(truth_array)
    if x.shape != y.shape or not np.isfinite(y).all() or (y < 0).any():
        raise ValueError('真值需与 dropout 同形、有限、非负；并请确认细胞/基因顺序一致')
    if expression_digest(x.astype(np.float32)) != result['input_hash']:
        raise ValueError('评估输入与训练输入不一致')
    masks = {'all': np.ones(x.shape, bool), 'input_zero': x == 0,
             'true_dropped': (x == 0) & (y > 0), 'true_zero': (x == 0) & (y == 0),
             'high_dropped_>100': (x == 0) & (y > 100)}
    base = result['predictions']['OwnOnly'].astype(np.float64)
    summary, strata, changes = [], [], []
    d, z, obs = masks['true_dropped'], masks['true_zero'], x > 0
    for name, output in result['predictions'].items():
        p = np.asarray(output, dtype=np.float64)
        if p.shape != x.shape or not np.isfinite(p).all() or (p < 0).any():
            raise ValueError(name + ' 预测无效')
        metrics = {region: metric_values(p, y, mask) for region, mask in masks.items()}
        gain = float(np.sum((base[d]-y[d])**2-(p[d]-y[d])**2, dtype=np.float64)/x.size)
        cost = float(np.sum(p[z]**2-base[z]**2, dtype=np.float64)/x.size)
        # count 张量是 float32；保留最终观测值的原始精度，不改动任何模型的零位预测。
        observed_change = float(np.max(np.abs(p[obs]-x[obs]))) if obs.any() else 0.
        if not np.array_equal(p.astype(np.float32)[obs], x.astype(np.float32)[obs]):
            raise AssertionError(name + ' 改变了观测值')
        summary.append(dict(model=name, role='historical_reference' if name in result['reference_names'] else 'matched_V23',
                            PCC=metrics['all']['PCC'], RMSE=metrics['all']['RMSE'], L1=metrics['all']['L1'],
                            dropped_RMSE=metrics['true_dropped']['RMSE'], zero_RMSE=metrics['true_zero']['RMSE'],
                            high_RMSE=metrics['high_dropped_>100']['RMSE'],
                            recovery_gain=gain, zero_cost=cost, net_MSE_gain=gain-cost,
                            zero_fill_rate_gt1=float((p[z] > 1.).mean()) if z.any() else np.nan,
                            observed_max_change=observed_change))
        for region, values in metrics.items():
            strata.append(dict(model=name, region=region, **values))
        for region in ['input_zero', 'true_dropped', 'true_zero', 'high_dropped_>100']:
            mask = masks[region]; n = int(mask.sum())
            delta = p[mask]-base[mask]
            better = np.abs(p[mask]-y[mask]) < np.abs(base[mask]-y[mask])
            worse = np.abs(p[mask]-y[mask]) > np.abs(base[mask]-y[mask])
            changes.append(dict(model=name, region=region, n=n,
                                mean_delta=float(delta.mean()) if n else np.nan,
                                increase_fraction=float((delta > 1e-6).mean()) if n else np.nan,
                                decrease_fraction=float((delta < -1e-6).mean()) if n else np.nan,
                                improved_fraction=float(better.mean()) if n else np.nan,
                                harmed_fraction=float(worse.mean()) if n else np.nan))
    return dict(comparison_df=pd.DataFrame(summary), strata_df=pd.DataFrame(strata),
                correction_df=pd.DataFrame(changes))

def v31_prepare_base(raw, cfg):
    """Build the two-view SelfExpr graph and a frozen own-only predictor from dropout input only."""
    input_hash = expression_digest(raw.detach().cpu().numpy())
    graph, graph_meta = v23_choose_graph(raw, cfg, trained=None)
    random_graph = diagnostic_random_graph(graph, int(cfg['seed']) + 2200)
    real_t = prepare_recovery_graph(graph, raw, raw.device)
    random_t = prepare_recovery_graph(random_graph, raw, raw.device)
    own, own_scale, own_history, weak_pool = v23_train_own(raw, real_t, random_t, cfg)
    own.eval().requires_grad_(False)
    own_hash = frozen_model_hash(own)
    own_prediction = diagnostic_own_predict(own, raw, cfg)
    raw_array = raw.detach().cpu().numpy().astype(np.float32, copy=True)
    predictions = {
        'Input': raw_array.copy(),
        'OwnOnly': preserve_observed(own_prediction, raw_array),
    }
    print('Shared base ready | graph source:', graph_meta.get('source', 'fresh_two_view'),
          '| weak-zero pool:', weak_pool, '| own predictor frozen.')
    return dict(own_model=own, graph=graph, random_graph=random_graph,
                graph_meta=graph_meta, own_history=own_history, own_scale=own_scale,
                predictions=predictions, input_hash=input_hash, own_frozen_hash=own_hash,
                reference_names=[])



## V38 对照设置
A/B/C **除了 AE 重构监督不同，其他结构、epoch、随机种子、人工验证位置都一样**。

完整参考数据只在所有训练和预测结束后的评价单元读取。


In [ ]:
V38_MODE='full'  # full=每阶段正式轮数；smoke=仅验证执行路径，不能用于正式性能比较
V38_CONFIG=dict(CONFIG)
V38_CONFIG.update(correction_epochs=300, correction_lr=3e-4, gene_embedding=16,
                  decoder_hidden=128,pair_batch=8192,log_every=25,read_dim=32,read_hidden=64,
                  ae_observed_weight=0.15)
if V38_MODE=='smoke':
    V38_CONFIG.update(ae_epochs=2,selfexpr_max_steps=3,selfexpr_min_steps=2,
                      cl_warmup=0,cl_ramp_epochs=1,branch_epochs=2,correction_epochs=2,
                      log_every=1)
elif V38_MODE!='full':raise ValueError('V38_MODE must be full or smoke')
V38_FILTER=dict(seed=42,holdout_fraction=.08,positive_recall_target=.95,cell_neighbors=15,
                svd_components=64,neighbor_temperature=.20,gene_neighbors=20,
                min_gene_detections=8,min_gene_coobserved=6,
                min_cell_calibration_positives=40,depth_groups=5,
                min_group_calibration_positives=80)
# original v32 cell below contains a self-test referencing V33_FILTER; alias only for that self-test.
V33_FILTER=V38_FILTER
print('V38:',V38_MODE,'AE',V38_CONFIG['ae_epochs'],'Own',V38_CONFIG['branch_epochs'],
      'GAT/GeneRead',V38_CONFIG['correction_epochs'],'epochs each per candidate')


## 复用 V31 已定义的基因级邻居读取器

`V31GATCorrection(use_read=True)` 在本版真正训练，保留原 `use_read=False` 作为严格 GAT 基线。邻居信息仅由遮盖输入计算，查询目标细胞本身不作为邻居。下一格保留 V32 的筛选函数定义，以便最终对三种补值器应用**相同**细胞邻居掩码（无 GDCC）。

In [ ]:
import numpy as np


import pandas as pd


from scipy import sparse


def v28_prepare_graph(graph, n):
    w=sparse.csr_matrix(graph,dtype=np.float64).copy()
    if w.shape!=(n,n): raise ValueError('Graph shape mismatch')
    if not np.isfinite(w.data).all() or (w.data<0).any():
        raise ValueError('SelfExpr weights must be finite and nonnegative')
    w=w.maximum(w.T).tocsr(); w.setdiag(0); w.eliminate_zeros()
    return w


def v28_leiden(graph, cfg):
    try:
        import igraph as ig
        import leidenalg
    except ImportError as exc:
        raise ImportError('请先在 notebook 中运行：%pip install igraph leidenalg，然后重启内核。') from exc
    edges=sparse.triu(graph,k=1).tocoo()
    g=ig.Graph(n=graph.shape[0],edges=list(zip(edges.row.tolist(),edges.col.tolist())),directed=False)
    if not g.ecount(): return np.arange(graph.shape[0])
    part=leidenalg.find_partition(g,leidenalg.RBConfigurationVertexPartition,
         weights=edges.data.tolist(),resolution_parameter=float(cfg['resolution']),
         seed=int(cfg['seed']),n_iterations=-1)
    return np.asarray(part.membership,dtype=np.int64)


"""Matched GAT baseline versus a gene-conditioned observed-neighbor count reader.

Only dropout counts enter training. Original zeros are unlabeled for this head.
This is a transductive training comparison, not cross-fitted validation.
"""

def v31_advance_legacy_rng(dim):
    """Match V26's initialization stream without keeping/training its unused NAF block."""
    for ni, no in ((3*dim, 2*dim), (dim, dim), (dim, dim),
                   (dim, 2*dim), (dim, dim)):
        nn.Linear(ni, no)  # initialization only; discarded immediately


def v31_neighbor_index(graph, raw):
    """Row i lists OTHER cells sending to i; padding never counts as an observation."""
    w = graph.tocsr(copy=True).astype(np.float32)
    if w.shape != (len(raw), len(raw)):
        raise ValueError('Graph/input shape mismatch')
    w.setdiag(0); w.eliminate_zeros(); w.sort_indices()
    if not np.isfinite(w.data).all() or (w.data < 0).any():
        raise ValueError('Invalid graph weights')
    degree = np.diff(w.indptr)
    width = max(1, int(degree.max()))
    ids = np.zeros((len(raw), width), dtype=np.int64)
    valid = np.zeros((len(raw), width), dtype=bool)
    weights = np.zeros((len(raw), width), dtype=np.float32)
    for i in range(len(raw)):
        lo, hi = w.indptr[i:i+2]; k = hi-lo
        if k:
            ids[i, :k] = w.indices[lo:hi]
            valid[i, :k] = True
            weights[i, :k] = w.data[lo:hi] / w.data[lo:hi].sum()
    return dict(index=torch.as_tensor(ids, device=raw.device),
                valid=torch.as_tensor(valid, device=raw.device),
                weight=torch.as_tensor(weights, device=raw.device))


@torch.no_grad()
def v31_read_context(corrupted, neighbors):
    """All values and library sizes come from the CURRENT masked input, never targets."""
    library = corrupted.sum(1)
    positive = library[library > 0]
    median = positive.median() if len(positive) else library.new_tensor(1.)
    depth = torch.where(library > 0, library / median.clamp_min(1e-8),
                        torch.ones_like(library)).clamp_min(1e-8)
    return dict(counts=corrupted, depth=depth, neighbors=neighbors)


class V31GATCorrection(nn.Module):
    """GAT residual head; V32 only instantiates use_read=False."""
    def __init__(self, own, cfg, use_read=False):
        super().__init__()
        self.use_read = bool(use_read)
        dim = int(cfg['feature_dim']); heads = int(cfg['gat_heads'])
        if dim % heads:
            raise ValueError('feature_dim must be divisible by gat_heads')
        self.gat = GATv2Conv(dim, dim//heads, heads=heads, concat=True,
                            dropout=float(cfg['attention_dropout']),
                            add_self_loops=True, edge_dim=1)
        v31_advance_legacy_rng(dim)
        self.simple_graph_scale = nn.Parameter(torch.zeros(1))
        gd = int(cfg['gene_embedding']); hidden = int(cfg['decoder_hidden'])
        self.gene_embedding = nn.Embedding(own.head.out_features, gd)
        self.residual_decoder = nn.Sequential(
            nn.Linear(dim+gd+1, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))
        nn.init.normal_(self.gene_embedding.weight, std=.02)
        nn.init.zeros_(self.residual_decoder[-1].weight)
        nn.init.zeros_(self.residual_decoder[-1].bias)
        if self.use_read:
            rd = int(cfg['read_dim'])
            self.read_query = nn.Linear(dim, rd, bias=False)
            self.read_gene = nn.Linear(gd, rd, bias=False)
            self.read_key = nn.Linear(dim, rd, bias=False)
            self.read_norm = math.sqrt(rd)
            # 7 evidence features + the original decoder inputs.
            self.read_gate = nn.Sequential(nn.Linear(dim+gd+1+7, int(cfg['read_hidden'])),
                                           nn.GELU(), nn.Linear(int(cfg['read_hidden']), 1))
            nn.init.zeros_(self.read_gate[-1].weight)
            nn.init.zeros_(self.read_gate[-1].bias)

    def fuse_nodes(self, h, graph_t):
        if graph_t['edge_index'].numel():
            g = self.gat(h, graph_t['edge_index'], edge_attr=graph_t['edge_attr'])
        else:
            g = torch.zeros_like(h)
        return h + self.simple_graph_scale * g

    def read_pairs(self, fused, pairs, gene_features, scale, context):
        ci, gi = pairs[:, 0], pairs[:, 1]
        neighbors = context['neighbors']
        js = neighbors['index'][ci]
        valid = neighbors['valid'][ci]
        weight = neighbors['weight'][ci]
        values = context['counts'][js, gi[:, None]]
        observed = valid & (values > 0)
        positive_count = observed.sum(1).to(fused.dtype)
        has_observation = positive_count > 0
        support = (weight * observed.to(weight.dtype)).sum(1)

        # Gene-conditioned weights, restricted to observed neighbor values.
        query = torch.tanh(self.read_query(fused[ci]) + self.read_gene(gene_features))
        keys = self.read_key(fused)
        logits = (query[:, None, :] * keys[js]).sum(-1) / self.read_norm
        logits = logits + weight.clamp_min(1e-12).log()
        logits = logits.masked_fill(~observed, -torch.finfo(logits.dtype).max)
        # All-missing rows remain finite: softmax, then mask and renormalize.
        attention = torch.softmax(logits, dim=1) * observed.to(logits.dtype)
        attention = attention / attention.sum(1, keepdim=True).clamp_min(1e-8)

        # Convert neighbor counts to the query cell's observed library-depth scale.
        # Raw model input and count targets are unchanged. No log value is added to count.
        values = values * context['depth'][ci, None] / context['depth'][js]
        values = torch.where(observed, values, torch.zeros_like(values))
        mean = (attention * values).sum(1)
        variance = (attention * (values-mean[:, None]).square()).sum(1)
        std = (variance.clamp_min(0.) + 1e-8).sqrt()
        maximum = values.max(1)[0]
        fraction = positive_count / valid.sum(1).clamp_min(1).to(fused.dtype)
        features = torch.stack((torch.log1p(mean/scale), torch.log1p(std/scale),
                                support, fraction, torch.log1p(positive_count),
                                torch.log1p(maximum/scale),
                                torch.log1p(context['depth'][ci])), dim=1)
        return mean, features, has_observation, support

    def decode_pairs(self, fused, pairs, base, gene_scale, context=None):
        ci, gi = pairs[:, 0], pairs[:, 1]
        scale = gene_scale[gi].clamp_min(.1)
        gene = self.gene_embedding(gi)
        inputs = torch.cat((fused[ci], gene, torch.log1p(base.clamp_min(0.)/scale)[:, None]), 1)
        delta = self.residual_decoder(inputs).squeeze(-1) * scale
        gate = torch.zeros_like(base)
        support = torch.zeros_like(base)
        has_observation = torch.zeros_like(base, dtype=torch.bool)
        if self.use_read:
            if context is None:
                raise ValueError('The gene reader requires the current masked input')
            mean, features, has_observation, support = self.read_pairs(fused, pairs, gene, scale, context)
            # A signed residual gain, NOT a dropout probability or a zero label.
            gate = torch.tanh(self.read_gate(torch.cat((inputs, features), 1)).squeeze(-1))
            gate = torch.where(has_observation, gate, torch.zeros_like(gate))
            delta = delta + gate * (mean-base)
        prediction = (base+delta).clamp_min(0.)
        return prediction, delta, dict(gate=gate, support=support, has_observation=has_observation)


def v31_mask_task(raw, cfg, epoch):
    # Same random stream as the old GAT baseline; mask only observed positives.
    with controlled_rng(int(cfg['seed'])*1000003 + 260000 + int(epoch), raw.device):
        hidden = (raw > 0) & (torch.rand_like(raw) < float(cfg['mask_ratio']))
        if not bool(hidden.any()):
            first = (raw > 0).nonzero(as_tuple=False)[0]
            hidden[first[0], first[1]] = True
        pairs = hidden.nonzero(as_tuple=False)
        target = raw[pairs[:, 0], pairs[:, 1]].detach()
        corrupted = raw.masked_fill(hidden, 0.)
    return dict(corrupted=corrupted, pairs=pairs, targets=target)


def v31_initialize_models(own, cfg, device):
    with controlled_rng(int(cfg['seed'])+2600, device):
        control = V31GATCorrection(own, cfg, False).to(device)
    with controlled_rng(int(cfg['seed'])+2600, device):
        reader = V31GATCorrection(own, cfg, True).to(device)
    # Explicit identical initialization of every common learned component.
    for name in ('gat', 'gene_embedding', 'residual_decoder'):
        getattr(reader, name).load_state_dict(getattr(control, name).state_dict())
    with torch.no_grad():
        reader.simple_graph_scale.copy_(control.simple_graph_scale)
    return {'GAT_Residual': control, 'GAT_GeneRead': reader}


@torch.no_grad()
def v31_predict(raw, own, model, graph_t, neighbors, cfg):
    own.eval(); model.eval()
    h = own.encode(raw)
    fused = model.fuse_nodes(h, graph_t)
    context = v31_read_context(raw, neighbors) if model.use_read else None
    output = raw.detach().cpu().numpy().copy()
    pairs = (raw == 0).nonzero(as_tuple=False)
    gate_sum = support_sum = covered = 0.
    for ps in slices(len(pairs), int(cfg['pair_batch'])):
        pair = pairs[ps]
        base = own.pairs(h, pair)
        pred, _, info = model.decode_pairs(fused, pair, base, own.gene_scale, context)
        output[pair[:, 0].cpu().numpy(), pair[:, 1].cpu().numpy()] = pred.cpu().numpy()
        gate_sum += float(info['gate'].abs().sum())
        support_sum += float(info['support'].sum())
        covered += float(info['has_observation'].sum())
    if not np.isfinite(output).all() or (output < 0).any():
        raise FloatingPointError('Nonfinite or negative predictions')
    return preserve_observed(output, raw.cpu().numpy()), dict(
        zero_positions=int(len(pairs)), read_coverage=covered/max(1, len(pairs)),
        mean_abs_read_gate=gate_sum/max(1, len(pairs)),
        mean_neighbor_support=support_sum/max(1, len(pairs)))


def v31_train(raw, cfg, base_artifact):
    """Only train the original GAT_Residual 300 epochs; do not modify GAT predictions."""
    input_hash = expression_digest(raw.detach().cpu().numpy())
    if input_hash != base_artifact['input_hash']:
        raise ValueError('Shared base does not match the current dropout input')
    own = base_artifact['own_model'].to(raw.device).eval().requires_grad_(False)
    graph = base_artifact['graph'].copy()
    graph_t = prepare_recovery_graph(graph, raw, raw.device)
    neighbors = v31_neighbor_index(graph, raw)
    own_hash, graph_hash = frozen_model_hash(own), graph_digest(graph)
    with controlled_rng(int(cfg['seed']) + 2600, raw.device):
        models = {'GAT_Residual': V31GATCorrection(own, cfg, use_read=False).to(raw.device)}
    optimizers = {n: torch.optim.Adam(m.parameters(), lr=float(cfg['correction_lr']))
                  for n, m in models.items()}
    loss_scale = float(raw[raw > 0].double().square().mean().clamp_min(1.).cpu())
    history = []
    with comparison_cpu_scope(raw.device, cfg):
        for epoch in range(1, int(cfg['correction_epochs'])+1):
            task = v31_mask_task(raw, cfg, epoch)
            pairs, targets = task['pairs'], task['targets']
            context = None  # GAT 基线不读取 GeneRead；不创建额外观测统计
            with torch.no_grad():
                h = own.encode(task['corrupted']).detach()
                base = own.pairs(h, pairs).detach()
            for name, model in models.items():
                started = time.perf_counter()
                model.train(); opt = optimizers[name]; opt.zero_grad(set_to_none=True)
                with controlled_rng(int(cfg['seed'])*1000003 + epoch*37 + 2601, raw.device):
                    fused = model.fuse_nodes(h, graph_t)
                leaf = fused.detach().requires_grad_(True)
                total = gate_sum = coverage = 0.
                for ps in slices(len(pairs), int(cfg['pair_batch'])):
                    pred, _, info = model.decode_pairs(leaf, pairs[ps], base[ps], own.gene_scale, context)
                    loss = (pred-targets[ps]).square().sum() / (len(pairs)*loss_scale)
                    if not bool(torch.isfinite(loss)):
                        raise FloatingPointError(name+' has nonfinite loss at epoch '+str(epoch))
                    loss.backward()
                    total += float(loss.detach())
                    gate_sum += float(info['gate'].detach().abs().sum())
                    coverage += float(info['has_observation'].sum())
                if leaf.grad is not None:
                    fused.backward(leaf.grad)
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg['grad_clip']),
                                                     error_if_nonfinite=True)
                opt.step(); sync_device(raw.device)
                history.append(dict(model=name, epoch=epoch, masked_positive_count=int(len(pairs)),
                    scaled_MSE=total, grad_norm=float(norm),
                    simple_graph_scale=float(model.simple_graph_scale.detach()),
                    read_gate_abs_mean=gate_sum/len(pairs), read_coverage=coverage/len(pairs),
                    seconds=time.perf_counter()-started))
            if epoch == 1 or epoch % int(cfg['log_every']) == 0 or epoch == cfg['correction_epochs']:
                last = {n: round(next(r['scaled_MSE'] for r in reversed(history) if r['model']==n), 6)
                        for n in models}
                print('V31 %d/%d | same masked targets=%d | MSE/scale=%s' %
                      (epoch, cfg['correction_epochs'], len(pairs), last))
            del task, context, h, base, fused, leaf
        predictions = dict(base_artifact['predictions']); diagnostics = []
        for name, model in models.items():
            model.eval().requires_grad_(False)
            pred, diag = v31_predict(raw, own, model, graph_t, neighbors, cfg)
            predictions[name] = pred
            diagnostics.append(dict(model=name, **diag))
    if frozen_model_hash(own) != own_hash or graph_digest(graph) != graph_hash:
        raise AssertionError('Shared OwnOnly or graph changed')
    if expression_digest(raw.detach().cpu().numpy()) != input_hash:
        raise AssertionError('Training changed the raw input')
    print('GAT_Residual training complete; OwnOnly and SelfExpr graph stayed frozen.')
    return dict(predictions=predictions, models=models, own_model=own, graph=graph,
                history=pd.DataFrame(history), read_diagnostics=pd.DataFrame(diagnostics),
                input_hash=input_hash, reference_names=[], config=dict(cfg),
                graph_meta=base_artifact['graph_meta'])




## 原 V32 细胞邻居筛选接口（仅作为共同后处理对照）

三组的原始预测首先**不做筛选**直接比较。随后可以对三组应用同一个固定的 V32 细胞邻居掩码，区分“补值幅度改进”和“补值位置筛选”的贡献；不使用基因共表达联合掩码来选模型。

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from scipy import sparse
import numpy as np
import pandas as pd

def v32_binary(raw):
    x = np.asarray(raw)
    if x.ndim != 2 or not np.isfinite(x).all() or (x < 0).any():
        raise ValueError('需要非负且有限的细胞×基因 count')
    return sparse.csr_matrix((x > 0).astype(np.float32))

def v32_cell_evidence_graph(binary, cfg):
    """从当前可见检出记录独立建图；不看目标值、不含自环、不根据目标基因挑选邻居。"""
    n, g = binary.shape
    if n < 3 or g < 3:
        raise ValueError('至少需要 3 个细胞与 3 个基因')
    weighted = TfidfTransformer(norm='l2').fit_transform(binary)
    dims = min(int(cfg['svd_components']), n - 1, g - 1)
    features = TruncatedSVD(n_components=dims, random_state=int(cfg['seed'])).fit_transform(weighted)
    features = normalize(features, norm='l2')
    k = min(int(cfg['cell_neighbors']), n - 1)
    # n_neighbors=k+1，一定剔除自身，必要时多取以防重复/近零向量导致自身位置不在首位。
    search_n = min(n, k+8)
    nn = NearestNeighbors(n_neighbors=search_n, metric='cosine', algorithm='brute', n_jobs=1)
    nn.fit(features)
    dist, ids = nn.kneighbors(features)
    rows, cols, vals = [], [], []
    for i in range(n):
        idx = ids[i][ids[i] != i][:k]
        if len(idx) == 0:
            continue
        chosen_dist = np.maximum(0., np.minimum(2., dist[i][ids[i] != i][:k]))
        logits = (1. - chosen_dist) / float(cfg['neighbor_temperature'])
        w = np.exp(logits - logits.max())
        w = w / w.sum()
        rows.extend([i]*len(idx))
        cols.extend(idx.tolist())
        vals.extend(w.tolist())
    graph = sparse.csr_matrix((np.array(vals,dtype=np.float32),(rows,cols)),shape=(n,n))
    graph.setdiag(0)
    graph.eliminate_zeros()
    norm = np.asarray(graph.sum(axis=1)).ravel()
    graph = sparse.diags(1./np.maximum(norm,1e-12)) @ graph
    assert graph.diagonal().sum() == 0
    return graph.tocsr()

def v32_gene_coexpression(binary, cfg):
    """二值检出 Phi 相关性，仅保留共检出足够多的正关联基因；禁止目标基因自关联。"""
    n, g = binary.shape
    count = np.asarray(binary.sum(axis=0), dtype=np.float64).ravel()
    detect_p = count / n
    cross = (binary.T @ binary).toarray().astype(np.float64)
    denom = np.sqrt(np.maximum(detect_p*(1.-detect_p),0.))
    denom = denom[:,None]*denom[None,:]
    phi = (cross/n - detect_p[:,None]*detect_p[None,:]) / np.maximum(denom,1e-10)
    enough = ((count >= float(cfg['min_gene_detections'])) &
              (count <= (n-float(cfg['min_gene_detections']))))
    phi[(cross < int(cfg['min_gene_coobserved'])) | (~enough[:,None]) | (~enough[None,:])] = 0.
    phi[~np.isfinite(phi)] = 0.
    np.fill_diagonal(phi,0.)
    phi = np.maximum(phi,0.)
    k = min(int(cfg['gene_neighbors']),g-1)
    rows, cols, weights = [],[],[]
    for gi in range(g):
        valid = np.flatnonzero(phi[gi] > 0)
        if not len(valid):
            continue
        if len(valid) > k:
            valid=valid[np.argpartition(phi[gi,valid],-k)[-k:]]
        w=phi[gi,valid].copy()
        w=w / w.sum()
        rows.extend([gi]*len(valid))
        cols.extend(valid.tolist())
        weights.extend(w.tolist())
    rel=sparse.csr_matrix((np.array(weights,dtype=np.float32),(rows,cols)),shape=(g,g))
    rel.setdiag(0)
    rel.eliminate_zeros()
    assert rel.diagonal().sum() == 0
    return rel

def v32_compute_support(raw,cfg):
    B=v32_binary(raw)
    cell_graph=v32_cell_evidence_graph(B,cfg)
    gene_graph=v32_gene_coexpression(B,cfg)
    # W_cell @ B：每个邻居按预先学习的图权重计入分母；零邻居不会被丢弃。
    neighbor_support=(cell_graph @ B).toarray().astype(np.float32)
    # B @ W_gene.T：同一细胞中与目标基因关联的其他基因的加权检出比例。
    gene_support=(B @ gene_graph.T).toarray().astype(np.float32)
    neighbor_support=np.clip(neighbor_support,0.,1.)
    gene_support=np.clip(gene_support,0.,1.)
    return dict(neighbor=neighbor_support, gene=gene_support,
                joint=np.minimum(neighbor_support,gene_support),
                graph_edges=int(cell_graph.nnz),gene_edges=int(gene_graph.nnz),
                gene_without_positive_links=int(np.sum(np.diff(gene_graph.indptr)==0)),
                detected_per_cell=np.asarray(B.sum(axis=1)).ravel().astype(np.int64))

def v32_quantile_lower(v, q):
    v=np.asarray(v,dtype=np.float64)
    if len(v)==0:
        raise ValueError('未找到用于校准的人工隐藏正例')
    try:
        return float(np.quantile(v,q,method='lower'))
    except TypeError:
        return float(np.quantile(v,q,interpolation='lower'))

def v32_calibrate(score, hidden, detect_counts, cfg):
    """使用人工留出的已知非零正例选择分数阈值，不读取原始零的负标签。"""
    cell, gene=np.nonzero(hidden)
    values=np.asarray(score[cell,gene],dtype=np.float64)
    recall_target=float(cfg['positive_recall_target'])
    q=1.-recall_target
    global_threshold=v32_quantile_lower(values,q)
    groups=int(cfg['depth_groups'])
    edges=np.unique(np.quantile(detect_counts, np.linspace(0.,1.,groups+1)))
    group_ids=np.searchsorted(edges[1:-1], detect_counts, side='right')
    group_thresholds={}
    positive_counts=np.bincount(group_ids[cell],minlength=max(1,len(edges)-1))
    for grp in range(max(1,len(edges)-1)):
        local=values[group_ids[cell]==grp]
        group_thresholds[grp]=(v32_quantile_lower(local,q)
             if len(local)>=int(cfg['min_group_calibration_positives']) else global_threshold)
    cell_counts=np.bincount(cell,minlength=score.shape[0])
    threshold=np.array([group_thresholds[int(g)] for g in group_ids],dtype=np.float32)
    for i in np.flatnonzero(cell_counts>=int(cfg['min_cell_calibration_positives'])):
        threshold[i]=v32_quantile_lower(values[cell==i],q)
    recall=float(np.mean(score[cell,gene] >= threshold[cell]))
    return dict(threshold=threshold,global_threshold=global_threshold,
        calibration_recall=recall, calibrating_positives=int(len(cell)),
        per_cell_threshold_count=int(np.sum(cell_counts>=int(cfg['min_cell_calibration_positives']))),
        depth_bin_edges=edges,group_thresholds=group_thresholds,group_ids=group_ids,
        group_positive_counts=positive_counts)

def v32_apply_threshold(calibration, current_counts):
    edges=calibration['depth_bin_edges']
    groups=np.searchsorted(edges[1:-1],current_counts,side='right')
    thresholds=np.array([calibration['group_thresholds'][int(g)] for g in groups],dtype=np.float32)
    # 每个细胞有足够多留出样本的情况，保留其单独的校准阈值。
    custom_count=int(calibration['per_cell_threshold_count'])
    # 由原校准阶段保留精确的阈值及 cell 是否单独校准标志。
    personal=calibration.get('personal_mask')
    if personal is not None:
        thresholds[personal]=calibration['threshold'][personal]
    return thresholds

def v32_build_filter(raw,cfg):
    """固定一次随机人工隐藏：先隐藏，再在隐藏后的可见输入上重新建证据图与基因图。"""
    original=np.asarray(raw,dtype=np.float32)
    observed=original>0
    rng=np.random.default_rng(int(cfg['seed'])+32000)
    hidden=observed & (rng.random(original.shape) < float(cfg['holdout_fraction']))
    if not hidden.any():
        raise ValueError('没有抽到可用的人工隐藏正例')
    masked=original.copy()
    masked[hidden]=0.
    assert not np.any(masked[hidden]>0)
    masked_hash=expression_digest(masked)
    calibration_support=v32_compute_support(masked,cfg)
    if expression_digest(masked)!=masked_hash:
        raise AssertionError('校准输入被意外改写')
    calibration={}
    for name in ('neighbor','joint'):
        cal=v32_calibrate(calibration_support[name],hidden,calibration_support['detected_per_cell'],cfg)
        cell_counts=np.bincount(np.nonzero(hidden)[0],minlength=len(original))
        cal['personal_mask']=cell_counts>=int(cfg['min_cell_calibration_positives'])
        calibration[name]=cal
    del masked,calibration_support
    inference=v32_compute_support(original,cfg)
    masks={}
    for name in ('neighbor','joint'):
        th=v32_apply_threshold(calibration[name],inference['detected_per_cell'])
        masks[name]=(inference[name]>=th[:,None]) & ~observed
        calibration[name]['inference_threshold']=th
    report=[]
    for name in ('neighbor','joint'):
        cal=calibration[name]
        chosen=masks[name]
        report.append(dict(filter=name,
            target_positive_recall=cfg['positive_recall_target'],
            masked_positive_recall=cal['calibration_recall'],
            masked_positive_count=cal['calibrating_positives'],
            zero_selected=int(chosen.sum()),
            zero_total=int((~observed).sum()),
            zero_selected_fraction=float(chosen.sum()/max(1,(~observed).sum())),
            thresholds_equal_zero=int(np.sum(cal['inference_threshold']<=0)),
            personal_cell_thresholds=cal['per_cell_threshold_count'],
            global_threshold=cal['global_threshold']))
    print('Calibration evidence graph re-estimated AFTER hiding known nonzeros.')
    print('Inference graph edges:',inference['graph_edges'],
          '| gene associations:',inference['gene_edges'],
          '| genes with no positive neighbors:',inference['gene_without_positive_links'])
    return dict(masks=masks,calibration=calibration,report=pd.DataFrame(report),
                evidence=inference,holdout_positive_count=int(hidden.sum()))

def v32_filter_prediction(raw, base, selected):
    raw=np.asarray(raw)
    base=np.asarray(base)
    if raw.shape!=base.shape or selected.shape!=raw.shape:
        raise ValueError('掩码或预测矩阵形状不一致')
    zero=raw==0
    out=raw.copy()
    out[zero & selected]=np.maximum(0.,base[zero & selected])
    assert np.array_equal(out[raw>0],raw[raw>0])
    return out

# 小型检查：检查图没有自环、两个证据分数范围及掩码尊重观测值。
_rng=np.random.default_rng(2)
_demo=_rng.poisson(1.8,size=(18,24)).astype(np.float32)
_demo[_rng.random(_demo.shape)<.5]=0
_quick=dict(V33_FILTER, svd_components=5, cell_neighbors=4, gene_neighbors=5,
            min_gene_detections=2, min_gene_coobserved=2,
            min_cell_calibration_positives=3,min_group_calibration_positives=4)
_demo_output=v32_build_filter(_demo,_quick)
for _s in ('neighbor','gene','joint'):
    assert np.isfinite(_demo_output['evidence'][_s]).all()
    assert (_demo_output['evidence'][_s]>=0).all() and (_demo_output['evidence'][_s]<=1.+1e-6).all()
assert not np.any(_demo_output['masks']['joint'] & (_demo>0))
print('V32 filter smoke test passed; main training will use full dataset.')


In [ ]:
# Train a SINGLE V33 GeneRead model; no old high-expression sampler or extra networks.
def v38_train_reader(raw,cfg,base):
    own=base['own_model'].to(raw.device).eval().requires_grad_(False)
    graph=base['graph'].copy()
    graph_t=prepare_recovery_graph(graph,raw,raw.device)
    neighbors=v31_neighbor_index(graph,raw)
    with controlled_rng(int(cfg['seed'])+2600,raw.device):
        model=V31GATCorrection(own,cfg,use_read=True).to(raw.device)
    opt=torch.optim.Adam(model.parameters(),lr=float(cfg['correction_lr']))
    scale=float(raw[raw>0].double().square().mean().clamp_min(1.).cpu())
    hist=[]
    with comparison_cpu_scope(raw.device,cfg):
        for epoch in range(1,int(cfg['correction_epochs'])+1):
            task=v31_mask_task(raw,cfg,epoch)
            pairs,targets=task['pairs'],task['targets']
            with torch.no_grad():
                h=own.encode(task['corrupted']).detach()
                baseline=own.pairs(h,pairs).detach()
            context=v31_read_context(task['corrupted'],neighbors)
            opt.zero_grad(set_to_none=True);model.train()
            with controlled_rng(int(cfg['seed'])*1000003+epoch*37+2601,raw.device):
                fused=model.fuse_nodes(h,graph_t)
            leaf=fused.detach().requires_grad_(True)
            total=0.
            for sl in slices(len(pairs),int(cfg['pair_batch'])):
                pred,delta,info=model.decode_pairs(leaf,pairs[sl],baseline[sl],own.gene_scale,context)
                loss=(pred-targets[sl]).square().sum()/(len(pairs)*scale)
                if not bool(torch.isfinite(loss)):raise FloatingPointError('V38 GeneRead loss nonfinite')
                loss.backward();total+=float(loss.detach())
            if leaf.grad is not None:fused.backward(leaf.grad)
            torch.nn.utils.clip_grad_norm_(model.parameters(),float(cfg['grad_clip']),error_if_nonfinite=True)
            opt.step()
            hist.append(dict(epoch=epoch,train_masked_positive=int(len(pairs)),scaled_MSE=total))
            if epoch==1 or epoch%int(cfg['log_every'])==0 or epoch==cfg['correction_epochs']:
                print('V38 GeneRead %d/%d | scaled MSE %.6f'%(epoch,cfg['correction_epochs'],total))
            del task,context,h,baseline,leaf,fused
        model.eval().requires_grad_(False)
        predicted,info=v31_predict(raw,own,model,graph_t,neighbors,cfg)
    return dict(prediction=predicted,history=pd.DataFrame(hist),diagnostics=info)


def v38_fit_pipeline(raw,cfg,mode):
    local=dict(cfg,ae_loss_mode=mode)
    with comparison_cpu_scope(raw.device,local):
        base=v31_prepare_base(raw,local)  # 300 AE + SelfExpr + OwnOnly, uses raw only
        gat=v31_train(raw,local,base)     # GAT 300, uses raw only
        reader=v38_train_reader(raw,local,base) # GeneRead 300, uses raw only
    pred=preserve_observed(reader['prediction'],raw.detach().cpu().numpy())
    return dict(prediction=pred,ae_mode=mode,graph_meta=base['graph_meta'],
                graph_hash=graph_digest(base['graph']),
                own_history=base['own_history'],gat_history=gat['history'],
                reader_history=reader['history'],reader_diagnostics=reader['diagnostics'])


def v38_holdout_validation(counts,seed=42,holdout_fraction=.06):
    original=np.asarray(counts,dtype=np.float32)
    if original.ndim!=2 or not np.isfinite(original).all() or (original<0).any():
        raise ValueError('input must be nonnegative finite 2D count matrix')
    observed=original>0
    rng=np.random.default_rng(int(seed)+38000)
    held=observed&(rng.random(original.shape)<float(holdout_fraction))
    if not held.any():raise ValueError('no validation positives sampled')
    train=original.copy();train[held]=0.
    if np.any(train[held]!=0.):raise AssertionError('held out values leaked')
    return train,held


def v38_val_metrics(pred,reference_input,held):
    y=np.asarray(reference_input)[held].astype(np.float64)
    p=np.asarray(pred)[held].astype(np.float64)
    rmse=float(np.sqrt(np.mean((p-y)**2)))
    l1=float(np.mean(np.abs(p-y)))
    pcc=float(np.corrcoef(p,y)[0,1]) if len(y)>1 and p.std()>0 and y.std()>0 else np.nan
    high=y>np.quantile(y,.95) # measured ONLY from held-out known positives
    h_rmse=float(np.sqrt(np.mean((p[high]-y[high])**2))) if high.any() else np.nan
    return dict(validation_PCC=pcc,validation_RMSE=rmse,validation_L1=l1,
                validation_high_RMSE=h_rmse,validation_positions=int(len(y)),
                high_positions=int(high.sum()))

# Unit test: held-out validation positives MUST be removed before the first AE sees the input.
_demo=np.array([[0,1,2,4],[1,2,0,3],[0,1,5,3],[2,7,8,0]],dtype=np.float32)
_x,_h=v38_holdout_validation(_demo,42,.35)
assert np.array_equal(_x[_h],np.zeros(_h.sum(),dtype=np.float32))
assert np.array_equal(_demo[~_h],_x[~_h])
print('V38 input split unit test passed')


## 严格人工隐藏验证（先遮盖，再训练整个模型）

A/B/C 在同一个 dropout40 正值留出掩码上比较；**验证集目标既不进入 AE/图/OwnOnly/GAT/GeneRead，也不参与筛选阈值校准。** 训练时只看 `X_v38_fit`。三组的筛选均用 V32 同一规则，在遮盖后训练输入上独立校准。人工正例 RMSE 用于预选 AE，不能验证原始零位置误补，最终仍需跨数据集。


In [ ]:
# ===== V38: leakage-safe, pre-AE holdout selection =====
V38_MODES={'A_full':'原全矩阵MSE','B_masked':'仅临时遮盖非零MSE',
           'C_masked_observed':'遮盖非零MSE+弱观测正值约束'}
V38_train_counts,V38_holdout_mask=v38_holdout_validation(count_array,int(V38_CONFIG['seed']),.06)
print('V38 held-out observed positives:',int(V38_holdout_mask.sum()))
X_v38_fit=torch.as_tensor(np.ascontiguousarray(V38_train_counts,dtype=np.float32),device=model_device)
assert (X_v38_fit.detach().cpu().numpy()[V38_holdout_mask]==0).all()
# The FILTER also calibrates exclusively on FIT counts; it does not see held-out target values.
V38_fit_filter=v32_build_filter(V38_train_counts,V38_FILTER)
V38_neighbor_mask=V38_fit_filter['masks']['neighbor']
V38_runs={}
V38_rows=[]
for mode,description in V38_MODES.items():
    print('\n### AE MODE',mode,description,'###')
    run=v38_fit_pipeline(X_v38_fit,V38_CONFIG,mode)
    raw_prediction=run['prediction']
    filt_prediction=v32_filter_prediction(V38_train_counts,raw_prediction,V38_neighbor_mask)
    assert np.array_equal(raw_prediction[V38_train_counts>0],V38_train_counts[V38_train_counts>0])
    assert np.array_equal(filt_prediction[V38_train_counts>0],V38_train_counts[V38_train_counts>0])
    for variant,pred in [('raw',raw_prediction),('filtered',filt_prediction)]:
        m=v38_val_metrics(pred,count_array,V38_holdout_mask)
        m.update(ae_mode=mode,prediction_type=variant,
                 filtered_holdout_recall=float(V38_neighbor_mask[V38_holdout_mask].mean()) if variant=='filtered' else 1.,
                 selected_zero_fraction=float(V38_neighbor_mask[V38_train_counts==0].mean()) if variant=='filtered' else 1.)
        V38_rows.append(m)
    V38_runs[mode]=dict(ae_mode=mode,graph_hash=run['graph_hash'],
                       validation_raw=raw_prediction,validation_filtered=filt_prediction,
                       own_history=run['own_history'],gat_history=run['gat_history'],
                       reader_history=run['reader_history'])
    print('Validation mode completed:',mode)
V38_validation=pd.DataFrame(V38_rows)
display(V38_validation)
# Rule fixed BEFORE looking at any true_data.csv: choose lowest artificial held-out filtered RMSE.
_rank=V38_validation[V38_validation.prediction_type=='filtered'].sort_values(
      ['validation_RMSE','validation_L1','ae_mode'],ascending=[True,True,True])
V38_selected_mode=str(_rank.iloc[0]['ae_mode'])
print('Selected using held-out dropout40 positives ONLY:',V38_selected_mode)

# ===== Final REFIT on complete dropout40: all stages retrained from scratch. =====
print('REFIT full dropout40 with selected AE mode:',V38_selected_mode)
V38_FINAL_RUN=v38_fit_pipeline(X_count,V38_CONFIG,V38_selected_mode)
V38_FINAL_FILTER=v32_build_filter(count_array,V38_FILTER)
V38_FINAL_MASK=V38_FINAL_FILTER['masks']['neighbor']
v38_final_raw=V38_FINAL_RUN['prediction']
final_imputed=v32_filter_prediction(count_array,v38_final_raw,V38_FINAL_MASK)
imputed_data=final_imputed
assert np.array_equal(final_imputed[count_array>0],count_array[count_array>0])
assert np.isfinite(final_imputed).all() and (final_imputed>=0).all()
# Save AFTER precommitted selection; neither mode choice nor thresholds ever reference true_data.csv.
from pathlib import Path
V38_OUTPUT_DIR=Path.cwd()/'v38_outputs'
V38_OUTPUT_DIR.mkdir(exist_ok=True)
np.savez_compressed(V38_OUTPUT_DIR/'final_prediction.npz',prediction=final_imputed,
                    baseline_raw=v38_final_raw,observed_input=count_array,
                    selected_mask=V38_FINAL_MASK.astype(np.uint8),
                    selected_mode=np.array(V38_selected_mode))
V38_validation.to_csv(V38_OUTPUT_DIR/'strict_masked_validation.csv',index=False)
print('V38 final selected AE mode:',V38_selected_mode)
print('Prediction:',V38_OUTPUT_DIR/'final_prediction.npz')
print('Artificially held-out validation:',V38_OUTPUT_DIR/'strict_masked_validation.csv')
print('Training Notebook never reads complete reference truth.')


## 最终评价（同一个 Notebook 的最后一步）

训练、人工遮盖验证、AE 方案选择、完整 dropout40 重训及 `final_imputed` 的生成**全部完成后**，下面才读取 `true_data.csv`。完整参考数据只用于报告 PCC、RMSE、L1、缺失非零位置、高表达位置和原始零位置指标；**禁止用这些评价结果重新选择模型、阈值或训练超参数**。

直接点击 **Kernel → Restart & Run All** 即可连续完成训练和评价，无需打开第二份 Notebook。

### 使用完整参考矩阵进行最终评价（训练已经结束）

这一步从磁盘读取刚刚保存的最终预测，避免评价数据进入前面的训练路径。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PREDICTED=V38_OUTPUT_DIR/'final_prediction.npz'
assert 'final_imputed' in globals() and 'V38_selected_mode' in globals(), '训练尚未完成，不能提前运行评价'
DATA_DIRS=[Path(r'E:/code/my/zeisel_data'),Path.cwd(),Path.cwd()/'zeisel_data',Path('/mnt/data')]
def find_path(filename):
    for d in DATA_DIRS:
        p=d/filename
        if p.is_file():return p
    raise FileNotFoundError(filename+' not found')
TRUE_PATH=find_path('true_data.csv')
DROPOUT_PATH=find_path('Zeisel_dropout_40_gene_cell.csv')
if not PREDICTED.is_file():raise FileNotFoundError(str(PREDICTED))
print('V38 train/selection complete:',V38_selected_mode)
print('Now opening true_data.csv for REPORT-ONLY evaluation:',TRUE_PATH)


In [ ]:
data=np.load(PREDICTED,allow_pickle=False)
pred=np.asarray(data['prediction'],dtype=np.float64)
raw=np.asarray(data['observed_input'],dtype=np.float64)
mask=np.asarray(data['selected_mask'],dtype=bool)
mode=str(data['selected_mode'])
true_frame=pd.read_csv(TRUE_PATH,index_col=0)
drop_frame=pd.read_csv(DROPOUT_PATH,index_col=0)
assert true_frame.index.equals(drop_frame.index) and true_frame.columns.equals(drop_frame.columns)
truth=np.asarray(true_frame.values.T,dtype=np.float64)
assert truth.shape==pred.shape==raw.shape
assert np.array_equal(drop_frame.values.T.astype(np.float32),raw.astype(np.float32))
assert np.array_equal(pred[raw>0],raw[raw>0])

def region_score(a,b,m):
    if not np.any(m):return {'n':0,'RMSE':np.nan,'L1':np.nan,'PCC':np.nan}
    p=a[m].astype(np.float64);t=b[m].astype(np.float64)
    rmse=float(np.sqrt(np.mean((p-t)**2)));l1=float(np.mean(np.abs(p-t)))
    pcc=float(np.corrcoef(p,t)[0,1]) if len(p)>1 and p.std()>0 and t.std()>0 else np.nan
    return {'n':int(len(p)),'RMSE':rmse,'L1':l1,'PCC':pcc}
regions={'all':np.ones(raw.shape,bool),'input_zeros':raw==0,
         'dropped_positive':(raw==0)&(truth>0),
         'reference_zeros':(raw==0)&(truth==0),
         'high_dropped_>100':(raw==0)&(truth>100)}
rows=[{'region':k,**region_score(pred,truth,v)} for k,v in regions.items()]
v38_final_metrics=pd.DataFrame(rows)
display(v38_final_metrics)
print('AE selection based on independent artificial positive holdout:',mode)
print('Selected original zeros:',int(mask.sum()),'of',int((raw==0).sum()))
print('Genuine dropped positive recall (evaluation only):',
      float(mask[(raw==0)&(truth>0)].mean()) if ((raw==0)&(truth>0)).any() else np.nan)
print('REFERENCE-ONLY: final_imputed and all model parameters were finalized before reading true_data.csv.')
